## 1. Configuración del Entorno de Trabajo
En esta celda se instalan las librerías necesarias para ejecutar el pipeline completo de ciencia de datos:

* **Manipulación de datos**: `pandas` y `numpy` para el manejo de estructuras tabulares y operaciones matriciales.
* **Machine Learning Tradicional**: `scikit-learn` para el preprocesamiento, métricas de evaluación y modelos base como Random Forest.
* **Modelos de Gradient Boosting**: `xgboost` y `lightgbm`, algoritmos de alto rendimiento para tareas de regresión.
* **Visualización**: `matplotlib` para la creación de gráficos de apoyo.

In [1]:
%pip install pandas numpy scikit-learn xgboost lightgbm

  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl.metadata (2.1 kB)
  Using cached lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl.metadata (17 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (8.9 MB)
Using cached xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl (131.7 MB)
Using cached lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl (3.6 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.2/300.2 MB 52.0 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [lightgbm]4/5 [lightgbm]arn]u12]
Note: you may need to restart the kernel to use updated packages.


## 2. Importación de Módulos y Modelos
En esta celda se cargan las funciones y clases específicas que se usarán en el pipeline:

* **Estructuras y cálculo:** `pandas` para manipular el dataset y `numpy` para operaciones matemáticas (como calcular la raíz cuadrada para el RMSE).
* **Modelos basados en árboles:** `RandomForestRegressor` (ensamblado clásico), `XGBoost` y `LightGBM` (algoritmos de gradient boosting de alto rendimiento).
* **Métricas de error:** `mean_absolute_error` (MAE) y `mean_squared_error` (MSE) para evaluar qué tan lejos están las predicciones de las ventas reales.

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

## 3. Carga y Agrupación Diaria
Esta celda importa el dataset y transforma la granularidad de los datos, pasando de registros de compra individuales a totales por día:

* **`read_csv(..., parse_dates=['Fecha'])`**: Carga el archivo y convierte la columna `Fecha` a formato *datetime* para que el sistema entienda que son fechas matemáticas, no texto.
* **`groupby('Fecha')[['TotalVenta']].sum()`**: Agrupa todas las transacciones ocurridas en el mismo día y suma sus importes para obtener la facturación total diaria.
* **`.reset_index()`**: Al aplicar el agrupamiento, Pandas convierte automáticamente la columna `Fecha` en el índice de la tabla. Este comando revierte esa acción, devolviendo Fecha a su estado de columna normal para poder operar con ella más adelante.
* **`.sort_values('Fecha')`**: Asegura el orden cronológico estricto, un paso innegociable en series temporales para que las variables de rezago (*lags*) no mezclen el futuro con el pasado.

In [3]:
df = pd.read_csv('../01.preprocesado_datos/dataset_limpio.csv', parse_dates=['Fecha'])
df_diario = df.groupby('Fecha')[['TotalVenta']].sum().reset_index().sort_values('Fecha')

print(df_diario)

         Fecha  TotalVenta
0   2010-12-01    46192.49
1   2010-12-02    47197.57
2   2010-12-03    23876.63
3   2010-12-05    31361.28
4   2010-12-06    31009.33
..         ...         ...
300 2011-12-05    58081.09
301 2011-12-06    45989.66
302 2011-12-07    69230.60
303 2011-12-08    50395.96
304 2011-12-09   184329.66

[305 rows x 2 columns]


Aquí tienes la documentación en Markdown para esa celda, incorporando la mejora técnica del **One-Hot Encoding** que discutimos para evitar errores de interpretación matemática en el modelo.

---

## 4. Ingeniería de Variables (Feature Engineering)

En esta etapa transformamos los datos brutos en información que los modelos de Machine Learning pueden procesar. Pasamos de tener solo una fecha y un valor a tener un contexto histórico para cada día:

### 1. Variables de Rezago (Lags)
* **`lag_1`**: Representa las ventas del día anterior. Permite al modelo capturar la inercia inmediata (si ayer se vendió mucho, hoy es probable que también).
* **`lag_7`**: Representa las ventas de hace exactamente una semana. Es fundamental para capturar la **estacionalidad semanal** (los domingos se parecen a los domingos anteriores).

### 2. Ventana Deslizante (Rolling Mean)
* **`media_7`**: Es el promedio de ventas de los últimos 7 días. Proporciona una visión de la tendencia reciente suavizando picos aislados. 
* *Nota técnica*: Se aplica `.shift(1)` antes del cálculo para evitar el **Data Leakage**. Esto garantiza que el modelo solo use información del pasado para intentar predecir el presente.

### 3. Codificación de Variable Categórica (One-Hot Encoding)
En lugar de dejar el día de la semana como un número del 0 al 6 (que confundiría al modelo haciéndole creer que el domingo vale más que el lunes), aplicamos `pd.get_dummies`:
* Transforma la columna `dia_semana` en múltiples columnas booleanas (0 o 1).
* **`drop_first=True`**: Elimina la primera columna generada para evitar la redundancia matemática. Si todos los días son 0, el modelo entiende automáticamente que es el día que fue eliminado, por eso se puede eliminar una columna. En este caso quitamos el dia_semana_0 (lunes). Tampoco hay dia_semana_5 porque ha dado la casualidad que no hay registros ese dia.

### 4. Limpieza de Datos
* **`dropna()`**: Al generar rezagos de 7 días, las primeras 7 filas del dataset quedan vacías (`NaN`). Eliminamos estas filas para que no generen errores durante el entrenamiento de los algoritmos.

In [4]:
df_diario['lag_1'] = df_diario['TotalVenta'].shift(1)
df_diario['lag_7'] = df_diario['TotalVenta'].shift(7)
df_diario['media_7'] = df_diario['TotalVenta'].shift(1).rolling(window=7).mean()
df_diario['dia_semana'] = df_diario['Fecha'].dt.dayofweek
df_diario = pd.get_dummies(df_diario, columns=['dia_semana'], drop_first=True)
df_diario = df_diario.dropna()

print(df_diario)

         Fecha  TotalVenta     lag_1     lag_7       media_7  dia_semana_1  \
7   2010-12-09    38193.91  39094.20  46192.49  38923.208571         False   
8   2010-12-10    33488.38  38193.91  47197.57  37780.554286         False   
9   2010-12-12    17102.35  33488.38  23876.63  35822.098571         False   
10  2010-12-13    27578.34  17102.35  31361.28  34854.344286         False   
11  2010-12-14    29235.13  27578.34  31009.33  34313.924286          True   
..         ...         ...       ...       ...           ...           ...   
300 2011-12-05    58081.09  20232.00  16969.25  38257.470000         False   
301 2011-12-06    45989.66  58081.09  51632.21  44130.590000          True   
302 2011-12-07    69230.60  45989.66  48640.57  43324.511429         False   
303 2011-12-08    50395.96  69230.60  41396.79  46265.944286         False   
304 2011-12-09   184329.66  50395.96  44405.37  47551.540000         False   

     dia_semana_2  dia_semana_3  dia_semana_4  dia_semana_6  
7

## 5. División del Dataset (Train, Validation y Test)

En problemas de series temporales, la división de datos debe ser **estrictamente cronológica**. No podemos usar el pasado para predecir el futuro si hemos "visto" el futuro durante el entrenamiento (evitamos el *Look-ahead bias*).

### Estrategia de Segmentación
* **Entrenamiento (Train):** Datos desde el inicio hasta el 30 de septiembre de 2011. Es el volumen principal para que el modelo aprenda la tendencia y estacionalidad.
* **Validación (Validation):** Mes de octubre de 2011. Se utiliza para evaluar el rendimiento de los modelos y realizar el ajuste de hiperparámetros sin tocar los datos finales de prueba.
* **Prueba (Test):** Noviembre de 2011 en adelante. Representa la simulación de "datos reales" que el modelo nunca ha visto, y será el conjunto que exportaremos a Power BI.



### Mejoras Técnicas Implementadas
1.  **Uso de `.copy()`**: Para no usar el dataframe original, hacemos una copia y evitamos posibles errores.
2.  **Features**: Hacemos la lista de `features` para que incluya todas las columnas.
3.  **Separación de Variables (X e y)**:
    * **$X$**: Características (Lags, Medias, Días de la semana).
    * **$y$**: Vector objetivo (Ventas reales a predecir).


In [5]:
# Añadimos .copy() para evitar SettingWithCopyWarning
train = df_diario[df_diario['Fecha'] < '2011-10-01'].copy()
val   = df_diario[(df_diario['Fecha'] >= '2011-10-01') & (df_diario['Fecha'] < '2011-11-01')].copy()
test  = df_diario[df_diario['Fecha'] >= '2011-11-01'].copy()

# Dinamizamos la selección de features tras el One-Hot Encoding
# Esto evita errores si faltan días (como los sábados)
cols_dia = []
for col in df_diario.columns:
    if 'dia_semana_' in col:
        cols_dia.append(col)

print (cols_dia)

        
features = ['lag_1', 'lag_7', 'media_7'] + cols_dia

# Creación de matrices de entrenamiento, validación y test
X_train, y_train = train[features], train['TotalVenta']
X_val, y_val     = val[features], val['TotalVenta']
X_test, y_test   = test[features], test['TotalVenta']

['dia_semana_1', 'dia_semana_2', 'dia_semana_3', 'dia_semana_4', 'dia_semana_6']


## 6. Entrenamiento y Comparación de Modelos

En esta fase, el objetivo es someter los datos a tres algoritmos de **Machine Learning** distintos para identificar cuál tiene mayor capacidad de predicción sobre el volumen de ventas. Utilizamos modelos basados en **ensambles de árboles**, que son el estándar de la industria para datos tabulares.

### 6.1. Algoritmos y Configuración de Parámetros
Para asegurar una competencia justa, hemos configurado los modelos con parámetros que equilibran la velocidad y la precisión:

* **Random Forest**: Construye múltiples árboles de decisión independientes y promedia sus predicciones.
    * `n_estimators=100`: Entrena 100 árboles para estabilizar el error.
    * `random_state=42`: Garantiza que los resultados sean idénticos en cada ejecución (reproducibilidad).
* **XGBoost & LightGBM**: Son modelos de *Gradient Boosting* que crean árboles de forma secuencial, donde cada uno corrige los errores del anterior.
    * `learning_rate=0.05`: Controla la velocidad de aprendizaje. Un valor bajo permite una convergencia más precisa, evitando el sobreajuste (*overfitting*).
    * `verbose=-1` (LightGBM): Limpia la salida de consola eliminando mensajes informativos secundarios.



### 6.2. Automatización del Flujo de Evaluación
En lugar de entrenar cada modelo manualmente, utilizamos un **bucle iterativo** que recorre un diccionario de modelos. Esto garantiza que todos reciban exactamente las mismas *features* y condiciones de entrenamiento:

1.  **Ajuste (`fit`)**: El modelo estudia la relación matemática entre las variables pasadas (lags, medias, días) y las ventas reales.
2.  **Predicción (`predict`)**: El modelo estima las ventas del periodo de **Validación** (Octubre 2011).
3.  **Cálculo de Métricas**: Comparamos la predicción con el valor real para medir el error.

### 6.3. Métricas de Error: ¿Cómo decidimos cuál es mejor?
No usamos una sola métrica, sino dos que se complementan:

* **MAE (Error Absoluto Medio)**: Es el error promedio en unidades de venta (euros). Es la métrica más realista para el negocio: "En promedio, el modelo se equivoca por X cantidad".
* **RMSE (Raíz del Error Cuadrático Medio)**: Esta métrica es más sensible a los errores grandes. Si el modelo falla estrepitosamente en un día de ventas récord, el RMSE subirá mucho más que el MAE.



### 6.4. Análisis Crítico de Selección
Tras ejecutar la comparativa, el modelo seleccionado para la exportación final a **Power BI** debe ser aquel que presente los valores de **MAE y RMSE más bajos** en el conjunto de validación. 

> **Observación Técnica**: En las pruebas realizadas, **LightGBM** suele mostrar una precisión superior y una mayor velocidad de ejecución. Es fundamental basar la elección en estas métricas objetivas y no en la popularidad del algoritmo, ya que un error menor en validación se traduce en una predicción más confiable para el negocio.

---

In [6]:
models = {
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds_val = model.predict(X_val)
    
    mae = mean_absolute_error(y_val, preds_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds_val))
    results.append({"Modelo": name, "MAE": mae, "RMSE": rmse})

# Mostrar comparativa
df_metrics = pd.DataFrame(results)
print("Resultados en Validación:\n", df_metrics)

Resultados en Validación:
          Modelo           MAE          RMSE
0  RandomForest  12016.348774  15474.027123
1       XGBoost  13875.222907  18734.099031
2      LightGBM  11079.187378  15397.395898


## 7. Predicción Final y Exportación para Power BI

Tras el análisis comparativo de modelos, procedemos a ejecutar la fase final del proyecto. Los resultados en validación determinaron que **LightGBM** es el algoritmo con mayor capacidad predictiva para este dataset, por lo que será el encargado de generar las estimaciones para el conjunto de prueba.

### 7.1. Aplicación del Mejor Modelo (LightGBM)
Utilizamos el modelo ganador para realizar la inferencia sobre el conjunto de **Test** (noviembre de 2011). 

* **Selección por métricas**: Se elige LightGBM por presentar el **MAE más bajo (11,079)** y el **RMSE más bajo (15,397)**, lo que garantiza el menor margen de error en las predicciones diarias.
* **Inferencia**: El modelo procesa las características (`X_test`) —que incluyen la memoria del negocio (*lags*) y el contexto semanal (*dummies*)— para proyectar las ventas.
* **Seguridad de datos**: Gracias al uso de `.copy()` en la división de datos, la columna de predicción se integra en el DataFrame de test sin disparar avisos de integridad de Pandas.

### 7.2. Preparación de la Salida para Business Intelligence
Para que los resultados sean consumibles por **Power BI**, realizamos una limpieza final del dataset de salida:

1.  **Selección de columnas críticas**: Filtramos el DataFrame para conservar únicamente la `Fecha`, la venta real (`TotalVenta`) y la `Prediccion` generada por el modelo.
2.  **Exportación a CSV**: Generamos el archivo `ventas_real_vs_predichas.csv` omitiendo el índice de filas para asegurar una estructura de tabla limpia y lista para importar.

### 7.3. Interpretación de Resultados para el Negocio
El archivo exportado permite visualizar el rendimiento del modelo en un entorno real:
* **Validación visual**: En Power BI se podrá contrastar la curva de ventas reales frente a las predichas para identificar desviaciones en días específicos.
* **Toma de decisiones**: La precisión lograda por LightGBM permite a la gerencia anticipar picos de demanda y planificar el inventario o el personal con una base científica, reduciendo la incertidumbre operativa.

In [7]:
# 5. Predicción final con el mejor modelo (ej. XGBoost) y exportación
best_model = models["LightGBM"] 
test['Prediccion'] = best_model.predict(X_test)

# Exportar para Power BI
output = test[['Fecha', 'TotalVenta', 'Prediccion']]
output.to_csv('ventas_real_vs_predichas.csv', index=False)